# Week 6 (starter): Advanced Retrieval, Transform and Measure

Extends Week 5. The transformation shown here is pseudo-relevance feedback (PRF), which is fully local and needs no key; HyDE is guarded behind a key. Cells marked **TODO (you)** are yours. Dependency: `sentence-transformers`. For HyDE: `pip install openai` and a Gemini key.

In [ ]:
import os, re, pathlib, numpy as np
from collections import Counter
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer, util
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# small labeled corpus (topic = relevance label). TODO (you): reuse YOUR Week 5 corpus and queries.
corpus = [('db','A B-tree index speeds up lookups by keeping keys sorted.'),('db','Sharding splits a large table across machines by a key.'),
  ('db','Transactions group writes so they commit or roll back atomically.'),('db','The query planner picks a join order to minimize cost.'),
  ('net','The TCP handshake opens a connection with SYN, SYN-ACK, ACK.'),('net','A load balancer spreads requests across backend servers.'),
  ('net','DNS resolves a domain name to an IP address.'),('net','Latency is the round-trip time for a packet between hosts.'),
  ('sec','TLS encrypts traffic so an eavesdropper cannot read it.'),('sec','SQL injection runs queries via unsanitized input.'),
  ('sec','A firewall filters traffic by port and address rules.'),('sec','Multi-factor auth adds a second proof beyond the password.'),
  ('cache','A CDN caches static assets near users to cut latency.'),('cache','Cache invalidation removes stale entries on change.'),
  ('cache','A time-to-live sets how long a cached entry stays valid.'),('cache','Write-through caching updates cache and store together.')]
topics=[c[0] for c in corpus]; docs=[c[1] for c in corpus]
doc_emb = embedder.encode(docs, convert_to_tensor=True, normalize_embeddings=True)
queries=[('db','how do indexes speed up reads'),('db','splitting a huge table across servers'),
  ('net','what happens when a TCP connection opens'),('sec','protect data so it cannot be read in transit'),
  ('cache','serve static files closer to users'),('net','why is everything so slow')]  # last is vague on purpose
K=4; REL=4
print('corpus', len(docs), 'queries', len(queries))

In [ ]:
STOP=set('a an the of to in on for and or is are how do does what so that with by as it why'.split())
def retrieve_idx(q,k=K):
    qe=embedder.encode([q],convert_to_tensor=True,normalize_embeddings=True)
    sims=util.cos_sim(qe,doc_emb).cpu().numpy()[0]; return list(np.argsort(sims)[::-1][:k])
def prf_expand(q, feedback_k=3, add=4):
    top=retrieve_idx(q,feedback_k); words=re.findall(r'[a-zA-Z]+',' '.join(docs[i] for i in top).lower())
    cand=[w for w in words if w not in STOP and len(w)>3 and w not in q.lower()]
    return q+' '+' '.join(w for w,_ in Counter(cand).most_common(add))
def precision(idxs,topic): return sum(1 for i in idxs if topics[i]==topic)/K

print('per-query precision@4, baseline -> PRF:')
improved=regressed=0
for topic,q in queries:
    b=precision(retrieve_idx(q),topic); a=precision(retrieve_idx(prf_expand(q)),topic)
    flag='IMPROVED' if a>b else ('REGRESSED' if a<b else 'same')
    improved+=a>b; regressed+=a<b
    print(f'  [{topic:5s}] {b:.2f} -> {a:.2f}  {flag}   {q!r}')
print(f'\nimproved {improved}, regressed {regressed}  (a regression is your required failure case)')
# TODO (you): also add a re-ranking step, and explain the regression.

## HyDE (needs a key) and submit
HyDE generates a hypothetical answer and retrieves on it. With a key it slots in where PRF is; without one it is skipped.

In [ ]:
def hyde(q, model='gemini-2.5-flash-lite'):
    if not os.environ.get('GEMINI_API_KEY'):
        return '[API-BLOCKED] set GEMINI_API_KEY to generate a hypothetical answer, then embed and retrieve on it'
    from openai import OpenAI
    c=OpenAI(api_key=os.environ['GEMINI_API_KEY'], base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return c.chat.completions.create(model=model, messages=[{'role':'user','content':'Write a one-sentence answer to: '+q}]).choices[0].message.content
print(hyde('how do indexes speed up reads'))
# Submit: PR with before/after deltas + a concrete cross-repo artifact with a classmate.
# Rubric: transform+rerank (30), before/after deltas (30), regression explained (20), cross-repo artifact (20).

## Part 1: Transform and re-rank
Extend your Week 5 retriever with at least one query transformation and a re-ranking step over the retrieved candidates.

## Part 2: Measure the effect
Run the same query set as Week 5 and report retrieval quality before and after, with per-query deltas. State which query types benefited and which did not.

## Part 3: Find one failure and explain it
Show at least one query where the transformation made retrieval worse, and explain why the technique misfired, since a transformation is an assumption about the query and assumptions break. If no query regressed, report that result and explain why the transformation held across your set.